# Exploratory Data Analysis (EDA)

This notebook explores the historical stock data downloaded via `yfinance`. It includes visualizations of price movements, volume, returns distributions, correlation analysis, and an assessment of the target class balance.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set consistent styling
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Add src directory to path so we can import our modules
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from data_loader import download_stock_data, create_target_variable

# Make sure reports directory exists
reports_dir = os.path.abspath(os.path.join('..', 'reports', 'figures'))
os.makedirs(reports_dir, exist_ok=True)

## 1. Data Loading & Overview
Load data for a sample ticker (e.g., AAPL) and check its core properties.

In [ ]:
TICKER = 'AAPL'
try:
    # Attempt to load from existing raw CSV first
    raw_data_path = os.path.join('..', 'data', 'raw', f'{TICKER}.csv')
    if os.path.exists(raw_data_path):
        df = pd.read_csv(raw_data_path, index_col=0, parse_dates=True)
        print(f"Loaded {TICKER} from existing CSV file.")
    else:
        df = download_stock_data(TICKER)
        print(f"Downloaded new {TICKER} data.")
except Exception as e:
    print(f"Error loading data: {e}")

# Add target variable
df = create_target_variable(df, horizon=1)

print(f"\nDataset Shape: {df.shape}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")

display(df.head())
display(df.tail())

## 2. Price History Plot
A clean line chart showing the closing price over the 4-year period.

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Close'], label='Closing Price', color='midnightblue', linewidth=1.5)

plt.title(f"{TICKER} Closing Price History", pad=15)
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend()
plt.tight_layout()

plt.savefig(os.path.join(reports_dir, f'{TICKER}_price_history.png'), dpi=300)
plt.show()

## 3. Volume Analysis
Visualize daily trading volume and its 20-day simple moving average (SMA).

In [ ]:
df['Volume_SMA_20'] = df['Volume'].rolling(window=20).mean()

fig, ax1 = plt.subplots(figsize=(14, 7))

ax1.bar(df.index, df['Volume'], color='steelblue', alpha=0.6, label='Daily Volume', width=1)
ax1.plot(df.index, df['Volume_SMA_20'], color='darkorange', linewidth=2, label='20-Day SMA Volume')

ax1.set_title(f"{TICKER} Trading Volume Analysis", pad=15)
ax1.set_xlabel("Date")
ax1.set_ylabel("Volume (Shares)")

# Format y-axis to show millions/billions for readability
def fmt_millions(x, pos):
    return f'{x/1e6:.0f}M'
ax1.yaxis.set_major_formatter(plt.FuncFormatter(fmt_millions))

ax1.legend()
plt.tight_layout()

plt.savefig(os.path.join(reports_dir, f'{TICKER}_volume_analysis.png'), dpi=300)
plt.show()

## 4. Returns Distribution
Log returns help stabilize variance and are a common starting point for financial time series modelling.

In [ ]:
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

plt.figure(figsize=(12, 6))
sns.histplot(df['Log_Return'].dropna(), bins=60, kde=True, color='teal')

plt.title(f"{TICKER} Daily Log Returns Distribution", pad=15)
plt.xlabel("Log Return")
plt.ylabel("Frequency")
plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='Zero Return')
plt.legend()
plt.tight_layout()

plt.savefig(os.path.join(reports_dir, f'{TICKER}_returns_distribution.png'), dpi=300)
plt.show()

print("Log Returns Summary Statistics:")
print(df['Log_Return'].describe())
print(f"\nSkewness: {df['Log_Return'].skew():.4f}")
print(f"Kurtosis: {df['Log_Return'].kurtosis():.4f}")

**Comment on Distribution:**
Financial returns often exhibit "fat tails" (leptokurtic distribution) and slight skewness compared to a perfect normal curve. The high kurtosis value (typically > 3) indicates that extreme events (large price swings) happen more frequently than a normal distribution would predict.

## 5. Rolling Statistics (Moving Averages)
Comparing short-term (20-day) vs medium-term (50-day) trends.

In [ ]:
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()

plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Close'], label='Close', alpha=0.5, color='gray')
plt.plot(df.index, df['SMA_20'], label='20-Day SMA', color='dodgerblue', linewidth=1.5)
plt.plot(df.index, df['SMA_50'], label='50-Day SMA', color='tomato', linewidth=1.5)

plt.title(f"{TICKER} Close Price & Moving Averages (20 & 50 Days)", pad=15)
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend()
plt.tight_layout()

plt.savefig(os.path.join(reports_dir, f'{TICKER}_moving_averages.png'), dpi=300)
plt.show()

## 6. Correlation Heatmap
Assess collinearity between basic price/volume features.

In [ ]:
cols_to_correlate = ['Open', 'High', 'Low', 'Close', 'Volume', 'Log_Return']
corr_matrix = df[cols_to_correlate].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, mask=mask, cmap='vlag', vmin=-1, vmax=1, center=0, 
            square=True, linewidths=.5, cbar_kws={"shrink": .8})

plt.title("Pearson Correlation Heatmap (OHLCV & Log Returns)", pad=15)
plt.tight_layout()

plt.savefig(os.path.join(reports_dir, f'{TICKER}_correlation_heatmap.png'), dpi=300)
plt.show()

## 7. Target Class Balance
Check if our binary classification target (1 = Up, 0 = Down) is imbalanced.

In [ ]:
class_counts = df['Target'].value_counts()
class_props = df['Target'].value_counts(normalize=True) * 100

plt.figure(figsize=(8, 6))
ax = sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')

plt.title("Target Variable Class Distribution", pad=15)
plt.xlabel("Class (0 = Down/Flat, 1 = Up)")
plt.ylabel("Count (Days)")
plt.xticks(ticks=[0, 1], labels=[f"Class 0\n({class_props[0]:.1f}%)", f"Class 1\n({class_props[1]:.1f}%)"])

# Add count labels on bars
for i, p in enumerate(ax.patches):
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points')

plt.tight_layout()
plt.savefig(os.path.join(reports_dir, f'{TICKER}_target_balance.png'), dpi=300)
plt.show()

**Comment on Class Imbalance:**
If the ratio is roughly 45%-55%, standard evaluation metrics like Accuracy are generally acceptable. If it is significantly skewed (e.g. 30%-70% in a strong bull market), we may need to employ class weighting in our models or use metrics like F1-Score or ROC-AUC for fair evaluation.

## 8. Key Observations for Feature Engineering

1. **High Multicollinearity among Price Features:** OHLC prices exhibit strong correlation (>0.99). Feeding all of them into classical linear models (like Logistic Regression) will cause instability. We should rely instead on derived price-ratio features (like returns, momentum indicators, or normalized oscillator distance) rather than absolute raw prices.

2. **Non-stationarity:** Absolute prices drift upward/downward over long periods, meaning statistical properties change over time. Classical ML models assume stationarity. We must transform absolute prices into relative indicators (Returns, Moving Average Crossovers, Relative Strength Index) to ensure stable feature distributions over time.

3. **Fat-tailed Returns Distributions:** Log returns display non-normal kurtosis. Standard scaling (z-score) is sensitive to extreme outliers, so we might need robust scalers or quantile transformations to handle extreme market volatility events (like earnings gaps or macro shocks).

4. **Class Balance Dependence on Market Regime:** The binary Target distribution (Up vs Down) may tilt towards Class 1 during sustained multi-year bull markets (e.g., AAPL). The evaluation phase should benchmark against the majority class baseline to ensure the model isn't just learning to statically predict "Up" every day.